# Fase 4 - Modelagem (FP-Growth restrito)

Regras **contexto -> desfecho_Fatal** via `src/mining.py`.

> `association_rules` tende a colocar Fatal no antecedente; usamos `gerar_regras_contexto_para_fatal()` a partir dos itemsets.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.config import (
    ANOS_OCORRENCIA,
    COLUNAS_CONTEXTO,
    DATA_DIR,
    DADOS_DIR,
    FIGURAS_DIR,
    FILTRO_UF,
    MIN_SUPPORT,
    PROCESSED_DIR,
    TABELAS_DIR,
)

plt.rcParams["figure.figsize"] = (11, 5)
sns.set_style("whitegrid")
FIGURAS_DIR.mkdir(parents=True, exist_ok=True)

import pickle
from src.mining import (
    classificar_estabilidade_temporal,
    comparar_uso_solo,
    minerar_por_ano,
    minerar_regras_fatais,
)
from src.evaluation import resumo_qualidade_regras, traduzir_regras

## 4.1 Carregamento

In [2]:
df_onehot = pd.read_pickle(PROCESSED_DIR / "transacional.pkl")
df_meta = pd.read_pickle(PROCESSED_DIR / "transacional_meta.pkl")
print(f"Matriz: {df_onehot.shape}")

Matriz: (26899, 78)


## 4.2 Mineracao contexto -> Fatal

In [3]:
itemsets, rules_all, rules_fatal = minerar_regras_fatais(df_onehot, min_support=MIN_SUPPORT)
rules_fatal = traduzir_regras(rules_fatal)
print(f"Itemsets: {len(itemsets):,} | Regras contexto->Fatal: {len(rules_fatal):,}")
print(resumo_qualidade_regras(rules_fatal))
if len(rules_fatal):
    display(rules_fatal[["antecedents_str", "support", "confidence", "lift", "explicacao_natural"]].head(15))

Itemsets: 14,005 | Regras contexto->Fatal: 76
{'n_regras': 76, 'lift_max': np.float64(5.4359), 'lift_mediano': np.float64(1.8086)}


,antecedents_str,support,confidence,lift,explicacao_natural
0,"ctx_fase_dia_Plena_Noite, ctx_tipo_acidente_At...",0.005316,0.425595,5.435938,Quando fase=dia Plena Noite E tipo=acidente At...
1,"ctx_causa_acidente_Transitar_na_contramão, ctx...",0.007398,0.422505,5.396472,Quando causa=acidente Transitar na contramão E...
2,"ctx_causa_acidente_Transitar_na_contramão, ctx...",0.007770,0.389199,4.971069,Quando causa=acidente Transitar na contramão E...
3,"ctx_condicao_metereologica_Céu_Claro, ctx_tipo...",0.011190,0.377193,4.817718,Quando condicao=metereologica Céu Claro E tipo...
4,"ctx_fase_dia_Plena_Noite, ctx_tipo_acidente_Co...",0.007398,0.376181,4.804798,Quando fase=dia Plena Noite E tipo=acidente Co...
5,"ctx_tipo_acidente_Colisão_frontal, ctx_tracado...",0.008662,0.375201,4.792279,Quando tipo=acidente Colisão frontal E tracado...
6,"ctx_fim_de_semana_Sim, ctx_tipo_acidente_Colis...",0.007919,0.373030,4.764543,Quando fim=de semana Sim E tipo=acidente Colis...
7,"ctx_causa_acidente_Transitar_na_contramão, ctx...",0.005056,0.372603,4.759089,Quando causa=acidente Transitar na contramão E...
8,"ctx_causa_acidente_Transitar_na_contramão, ctx...",0.008253,0.370618,4.733735,Quando causa=acidente Transitar na contramão E...
9,"ctx_fase_dia_Plena_Noite, ctx_tipo_acidente_At...",0.009108,0.367316,4.691568,Quando fase=dia Plena Noite E tipo=acidente At...


## 4.3 Estabilidade temporal

In [4]:
rules_por_ano = minerar_por_ano(df_onehot, df_meta, min_support=MIN_SUPPORT)
estabilidade = classificar_estabilidade_temporal(rules_por_ano)
print("Regras por ano:", {k: len(v) for k, v in rules_por_ano.items()})
if not estabilidade.empty:
    print(estabilidade["status"].value_counts())
    display(estabilidade.head(12))
with open(PROCESSED_DIR / "rules_por_ano.pkl", "wb") as f:
    pickle.dump(rules_por_ano, f)

Regras por ano: {2023: 85, 2024: 73, 2025: 77, 2026: 86}
status
estavel        87
transitoria    57
Name: count, dtype: int64


,regra,n_anos_presente,n_anos_total,freq_relativa,status,lift_medio,confianca_media,anos
0,"ctx_fase_dia_Plena_Noite, ctx_tipo_acidente_At...",2,4,0.50,estavel,5.6527,0.4371,"[2023, 2025]"
4,"ctx_causa_acidente_Transitar_na_contramão, ctx...",4,4,1.00,estavel,5.5566,0.4393,"[2023, 2024, 2025, 2026]"
6,"ctx_causa_acidente_Transitar_na_contramão, ctx...",4,4,1.00,estavel,5.0138,0.3959,"[2023, 2024, 2025, 2026]"
11,"ctx_fase_dia_Plena_Noite, ctx_tipo_acidente_Co...",4,4,1.00,estavel,4.9423,0.3904,"[2023, 2024, 2025, 2026]"
92,"ctx_fim_de_semana_Sim, ctx_tipo_acidente_Colis...",3,4,0.75,estavel,4.9066,0.3895,"[2024, 2025, 2026]"
88,"ctx_tipo_acidente_Colisão_frontal, ctx_tracado...",3,4,0.75,estavel,4.8900,0.3883,"[2024, 2025, 2026]"
2,"ctx_condicao_metereologica_Céu_Claro, ctx_tipo...",4,4,1.00,estavel,4.8476,0.3825,"[2023, 2024, 2025, 2026]"
5,"ctx_condicao_metereologica_Céu_Claro, ctx_fim_...",2,4,0.50,estavel,4.7893,0.3744,"[2023, 2024]"
12,"ctx_causa_acidente_Transitar_na_contramão, ctx...",4,4,1.00,estavel,4.7833,0.3777,"[2023, 2024, 2025, 2026]"
14,"ctx_condicao_metereologica_Céu_Claro, ctx_fase...",3,4,0.75,estavel,4.6937,0.3728,"[2023, 2024, 2026]"


## 4.4 Estratos urbano vs. rural (H5)

In [5]:
for estrato, rules in comparar_uso_solo(df_onehot, df_meta).items():
    print(f"\n=== {estrato.upper()} ({len(rules)} regras) ===")
    if len(rules):
        display(rules[["antecedents_str", "lift", "confidence"]].head(5))

C:\Users\erikr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\mlxtend\frequent_patterns\association_rules.py:186: RuntimeWarning: invalid value encountered in divide
  cert_metric = np.where(certainty_denom == 0, 0, certainty_num / certainty_denom)
C:\Users\erikr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\mlxtend\frequent_patterns\association_rules.py:186: RuntimeWarning: invalid value encountered in divide
  cert_metric = np.where(certainty_denom == 0, 0, certainty_num / certainty_denom)



=== URBANO (19 regras) ===


,antecedents_str,lift,confidence
0,"ctx_fase_dia_Plena_Noite, ctx_tipo_acidente_At...",8.427070,0.386076
1,"ctx_condicao_metereologica_Céu_Claro, ctx_fase...",6.936703,0.317797
2,"ctx_fase_dia_Plena_Noite, ctx_tipo_acidente_At...",6.726297,0.308157
3,"ctx_condicao_metereologica_Céu_Claro, ctx_tipo...",6.431315,0.294643
4,"ctx_fim_de_semana_Sim, ctx_tipo_acidente_Atrop...",5.930013,0.271676



=== RURAL (54 regras) ===


,antecedents_str,lift,confidence
0,"ctx_causa_acidente_Transitar_na_contramão, ctx...",4.762937,0.439535
1,"ctx_fase_dia_Plena_Noite, ctx_tipo_acidente_At...",4.611882,0.425595
2,"ctx_causa_acidente_Transitar_na_contramão, ctx...",4.578399,0.422505
3,"ctx_fim_de_semana_Sim, ctx_tipo_acidente_Colis...",4.407991,0.406780
4,"ctx_tipo_acidente_Atropelamento_de_Pedestre, c...",4.381469,0.404332


## 4.5 Exportacao

In [6]:
itemsets.to_csv(DADOS_DIR / "itemsets_frequentes.csv", index=False)
rules_all.to_csv(DADOS_DIR / "regras_completas.csv", index=False)
rules_fatal.to_csv(DADOS_DIR / "regras_contexto_fatal.csv", index=False)
rules_fatal.head(20).to_csv(TABELAS_DIR / "top20_regras_contexto_fatal.csv", index=False)
if not estabilidade.empty:
    estabilidade.to_csv(TABELAS_DIR / "regras_estabilidade_temporal.csv", index=False)
rules_fatal.to_pickle(PROCESSED_DIR / "rules_fatal.pkl")
print("[OK] Artefatos exportados")

[OK] Artefatos exportados
